In [1]:
import matplotlib.pyplot as plt
import numpy as np
from src.data import SimulatedData, Strategy
from src.matching import Matching
from src.evaluation import Evaluation
from tqdm import tqdm
from matplotlib.ticker import FormatStrFormatter
import random
import pandas as pd

In [2]:
def get_demand(agents, items):
    demands = {i:[] for i in items[0]}
    n_agents = len(agents[0])
    for idx, agent in enumerate(agents):
        for i in demands:
            count = 0
            for pref in agent.values():
                if pref[0] == i:
                    count += 1
            demands[i].append(count)
    avg_demand = {i:(np.average(demands[i])/n_agents,np.std(demands[i])/n_agents) for i in demands}
    return avg_demand

def get_lotteries(advies=None):
    # rsm denotes the reverse school map - we need this to track placement to "Not Placed" school
    agents, items, _, rsm = loader.load_preferences_capacities(advies=advies)
    not_placed_id = list(rsm.keys())[list(rsm.values()).index('Not Placed')]
    original_order = sorted(list(agents.keys()))
    shuffled_order = original_order.copy()
    random.shuffle(shuffled_order)
    lottery_map = dict(zip(original_order, shuffled_order))
    shuffled_agents = {lottery_map[l]:agents[l] for l in lottery_map}
    return shuffled_agents, items, not_placed_id

In [3]:
from src.data import SchoolChoiceData
YEAR = 2017
path_to_data = f"../../data/osvo/da_stb/v1/{YEAR}.xlsx"
loader = SchoolChoiceData(path=path_to_data)
school_df, student_df = loader.load_dataframes()

# filter out priority students
priority = student_df[student_df['Voorrang/Hardheid eerste voorkeur'] != '-']
priority = priority[priority['Voorkeur 1'] == priority['Geplaatst op']]
without_priority = pd.concat([student_df, priority]).drop_duplicates(keep=False)
# update capacities
for school in priority['Geplaatst op'].tolist():
    school_df.loc[school_df[school_df['Key'] == school].index, 'Maximale capaciteit definitieve matching'] -= 1
loader.set_dataframes(without_priority, school_df)

In [4]:
# N_AGENTS = 1000
# N_ITEMS = 10
RUNS = 50
SEED = 1610

scenario_labels = ["vwo", "havo/vwo", "vmbo"]
scenarios = {label:{} for label in scenario_labels}

In [5]:
for label in scenario_labels:
    scenarios[label]['agents'] = []
    scenarios[label]['items'] = []
    scenarios[label]['RSD'] = []
    scenarios[label]['RM-HAL'] = []

In [6]:
data = SimulatedData(seed=SEED)
topns = [1, 2, 3, "Mixed-n"]
fractions = [1.0, .75, .50, .25]
DEFAULT_TOPN = "Mixed-n"
DEFAULT_FRACTION = 1.0
DEFAULT_STRATEGY = "FOH"
CUTOFF = False

In [7]:
random.seed(SEED)
for label in scenario_labels:
    for run in tqdm(range(RUNS)):
        # generate simulated data
        agents, items, not_placed_id = get_lotteries(advies=label)
        scenarios[label]['agents'].append(agents)
        scenarios[label]['items'].append(items)
        scenarios[label]['Not Placed'] = not_placed_id
        # apply matching algorithms
        matching = Matching(agents=agents, items=items, not_placed_id=not_placed_id)
        # random priority
        scenarios[label]['RSD'].append(matching.random_priority())
        # hungarian algorithm
        scenarios[label]['RM-HAL'].append(matching.hungarian_algorithm())

        #strategy exp 1a
        strategy = Strategy(seed=SEED, agents=agents, items=items, not_placed_id=not_placed_id)
        for topn in topns:
            slabel = f'RM-HAL, n={topn}'
            alabel = f'agents, n={topn}'
            strategic_agents, strategic_agent_ids = strategy.get_strategic_agents(stype=DEFAULT_STRATEGY, fraction=DEFAULT_FRACTION, topn=topn, cutoff=CUTOFF)
            matching_s = Matching(agents=strategic_agents, items=items, not_placed_id=not_placed_id)
            allocation_ha_s = matching_s.hungarian_algorithm()
            if slabel not in scenarios[label]:
                scenarios[label][slabel] = []
            if alabel not in scenarios[label]:
                scenarios[label][alabel] = []
            scenarios[label][slabel].append(allocation_ha_s)
            scenarios[label][alabel].append((strategic_agents, strategic_agent_ids))

        #strategy exp 1b
        for fraction in fractions:
            slabel = f'RM-HAL, f={int(fraction*100)}%'
            flabel = f'agents, f={int(fraction*100)}%'
            strategic_agents, strategic_agent_ids = strategy.get_strategic_agents(stype=DEFAULT_STRATEGY, fraction=fraction, topn=DEFAULT_TOPN, cutoff=CUTOFF)
            matching_s = Matching(agents=strategic_agents, items=items, not_placed_id=not_placed_id)
            allocation_ha_s = matching_s.hungarian_algorithm()
            if slabel not in scenarios[label]:
                scenarios[label][slabel] = []
            if flabel not in scenarios[label]:
                scenarios[label][flabel] = []
            scenarios[label][slabel].append(allocation_ha_s)
            scenarios[label][flabel].append((strategic_agents, strategic_agent_ids))

        #strategy exp 2
        for topn in topns:
            slabel = f'RM-HAL, n={topn}'
            alabel = f'agents, n={topn} individual'
            strategic_agents, strategic_agent_ids = strategy.get_strategic_agents(stype=DEFAULT_STRATEGY, fraction=0.5, topn=topn, topn_mode="default", cutoff=CUTOFF)
            matching_s = Matching(agents=strategic_agents, items=items, not_placed_id=not_placed_id)

            allocation_ha_s = matching_s.hungarian_algorithm()
            allocation_ha_truth = {k:v for k,v in allocation_ha_s.items() if k not in strategic_agent_ids}
            allocation_ha_strat = {k:v for k,v in allocation_ha_s.items() if k in strategic_agent_ids}

            slabelt = slabel + ' truthful'
            slabelst = slabel + ' strategic'
            slabelb = slabel + ' baseline'

            if slabelt not in scenarios[label]:
                scenarios[label][slabelt] = []
            if slabelst not in scenarios[label]:
                scenarios[label][slabelst] = []
            if slabelb not in scenarios[label]:
                scenarios[label][slabelb] = []
            if alabel not in scenarios[label]:
                scenarios[label][alabel] = []

            scenarios[label][slabelt].append(allocation_ha_truth)
            scenarios[label][slabelst].append(allocation_ha_strat)
            scenarios[label][slabelb].append(allocation_ha_s)
            scenarios[label][alabel].append((strategic_agents, strategic_agent_ids))

100%|██████████| 50/50 [13:51<00:00, 16.63s/it]


In [8]:
import pickle
pickle.dump(scenarios, open(f'ams_data_seed_{SEED}_mixed_n.pkl', 'wb'))